# Inference for Regression

In [ ]:
library(tidyverse)
library(infer)
library(patchwork)
options(repr.matrix.max.rows = 20)

## Part 1: The Regression Model

A true line we can't see, plus random noise. `draw_and_compare()` is the function from the Inference for Regression module. Its arguments are the slope and intercept of the true line, and the number of points.

In [ ]:
draw_and_compare <- function(true_slope, true_int, sample_size) {
  
  # 1. Create some synthetic data
  x <- rnorm(sample_size, mean = 50, sd = 5)
  eps <- rnorm(sample_size, mean = 0, sd = 6)
  y <- true_slope * x + true_int + eps
  
  df <- tibble(x, y)
  xlims <- range(x)
  
  # 2. Fit regression line
  fit <- lm(y ~ x, data = df)
  coef_fit <- coef(fit)
  a_hat <- coef_fit[1]
  b_hat <- coef_fit[2]
  
  true_line <- tibble(x = xlims, y = true_slope * xlims + true_int)
  fit_line  <- tibble(x = xlims, y = a_hat + b_hat * xlims)
  
  p1 <- ggplot(df, aes(x, y)) +
    geom_point(alpha = 0.7) +
    geom_line(data = true_line, aes(x, y), color = "forestgreen", linewidth = 1) +
    labs(title = "True Line, and Points Created", x = NULL, y = NULL)
  
  p2 <- ggplot(df, aes(x, y)) +
    geom_point(alpha = 0.7) +
    labs(title = "What We Get to See", x = NULL, y = NULL)
  
  p3 <- ggplot(df, aes(x, y)) +
    geom_point(alpha = 0.7) +
    geom_line(data = fit_line, aes(x, y), color = "goldenrod2", linewidth = 1) +
    labs(title = "Regression Line: Estimate of True Line", x = NULL, y = NULL)
  
  p4 <- ggplot(df, aes(x, y)) +
    geom_point(alpha = 0.7) +
    geom_line(data = fit_line, aes(x, y), color = "goldenrod2", linewidth = 1) +
    geom_line(data = true_line, aes(x, y), color = "forestgreen", linewidth = 1) +
    labs(title = "Regression Line and True Line", x = NULL, y = NULL)
  
  (p1 / p2 / p3 / p4) & theme(plot.title = element_text(hjust = 0.5))
}

10 points from the true line with slope 2 and intercept −5. Run it more than once.

In [ ]:
options(repr.plot.width = 5, repr.plot.height = 12)
draw_and_compare(2, -5, 10)

In [ ]:
draw_and_compare(2, -5, 10)

Now 100 points.

In [ ]:
draw_and_compare(2, -5, 100)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 5)

## Part 2: Inference for the True Slope

Birth weight (ounces) and gestational days for 1,174 babies.

In [ ]:
baby <- read_csv("data/baby.csv", show_col_types = FALSE)
baby

In [ ]:
ggplot(baby, aes(x = `Gestational Days`, y = `Birth Weight`)) +
  geom_point(alpha = 0.5) +
  geom_smooth(method = "lm", se = FALSE, color = "dodgerblue3", linewidth = 1.2) +
  theme_minimal(base_size = 16)

Guess $r$.

In [ ]:
baby_r <- cor(baby$`Gestational Days`, baby$`Birth Weight`)
baby_r

In [ ]:
baby_slope <- baby_r * sd(baby$`Birth Weight`) / sd(baby$`Gestational Days`)
baby_slope

In [ ]:
baby_model <- lm(`Birth Weight` ~ `Gestational Days`, data = baby)
coef(baby_model)

### Bootstrap the scatter plot

Resample the rows with replacement, 5,000 times, and find the slope each time.

In [ ]:
set.seed(42)

boot_slopes <- baby |>
  specify(`Birth Weight` ~ `Gestational Days`) |>
  generate(reps = 5000, type = "bootstrap") |>
  calculate(stat = "slope")

boot_slopes

In [ ]:
slope_ci <- boot_slopes |>
  get_ci(level = 0.95, type = "percentile")

slope_ci

In [ ]:
ggplot(boot_slopes, aes(x = stat)) +
  geom_histogram(bins = 20, fill = "chartreuse4", color = "white") +
  geom_vline(xintercept = c(slope_ci$lower_ci, slope_ci$upper_ci),
             linetype = "dashed", linewidth = 1) +
  geom_vline(xintercept = baby_slope, color = "dodgerblue3", linewidth = 1) +
  labs(x = "Bootstrap slopes", y = "Count") +
  theme_minimal(base_size = 16)

## Part 3: Could the True Slope Be 0?

A true line with slope 0, and 25 points. Run it a few times.

In [ ]:
options(repr.plot.width = 5, repr.plot.height = 12)
draw_and_compare(0, 10, 25)

In [ ]:
draw_and_compare(0, 10, 25)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 5)

**Null hypothesis:** the slope of the true line is 0.

**Alternative hypothesis:** the slope of the true line is not 0.

If the 95% confidence interval for the true slope doesn't contain 0, reject the null hypothesis.

### Maternal age and birth weight

In [ ]:
ggplot(baby, aes(x = `Maternal Age`, y = `Birth Weight`)) +
  geom_point(alpha = 0.5) +
  geom_smooth(method = "lm", se = FALSE, color = "dodgerblue3", linewidth = 1.2) +
  theme_minimal(base_size = 16)

In [ ]:
coef(lm(`Birth Weight` ~ `Maternal Age`, data = baby))

In [ ]:
set.seed(42)

age_slopes <- baby |>
  specify(`Birth Weight` ~ `Maternal Age`) |>
  generate(reps = 5000, type = "bootstrap") |>
  calculate(stat = "slope")

age_slopes |>
  get_ci(level = 0.95, type = "percentile")

## Try it out!

For each predictor, find the slope and a 95% bootstrap confidence interval for the true slope. Do you reject the null hypothesis that the true slope is 0?

1. `Maternal Pregnancy Weight` (pounds)
2. `Maternal Smoker`. It's `TRUE` or `FALSE`. Convert it to 1 and 0 first with `as.numeric()`.

**Done early?** `Maternal Height` (inches)

In [ ]:
# 1. Maternal Pregnancy Weight

In [ ]:
# 2. Maternal Smoker

In [ ]:
# Done early? Maternal Height

## Part 4: Predicting the Height of the True Line

The regression line's estimate at 300 gestational days:

In [ ]:
predict(baby_model, newdata = tibble(`Gestational Days` = 300))

### Bootstrap lines

For each bootstrap sample, find the slope and intercept of its regression line.

In [ ]:
set.seed(42)

boot_lines <- baby |>
  specify(`Birth Weight` ~ `Gestational Days`) |>
  generate(reps = 5000, type = "bootstrap") |>
  group_by(replicate) |>
  summarize(
    slope = cor(`Gestational Days`, `Birth Weight`) * sd(`Birth Weight`) / sd(`Gestational Days`),
    intercept = mean(`Birth Weight`) - slope * mean(`Gestational Days`)
  )

boot_lines

The first 10 lines, from 230 to 310 days, with each line's estimate at 300 days. The dashed line is the average number of gestational days.

In [ ]:
boot_lines |>
  slice(1:10) |>
  ggplot() +
  geom_abline(aes(slope = slope, intercept = intercept, color = factor(replicate)), linewidth = 0.8) +
  geom_vline(xintercept = mean(baby$`Gestational Days`), linetype = "dashed") +
  geom_point(aes(x = 300, y = slope * 300 + intercept, color = factor(replicate)), size = 3.5) +
  coord_cartesian(xlim = c(230, 310), ylim = c(85, 140)) +
  labs(x = "Gestational Days", y = "Birth Weight") +
  guides(color = "none") +
  theme_minimal(base_size = 16)

The estimate at 300 days for all 5,000 lines:

In [ ]:
predictions_300 <- boot_lines |>
  mutate(stat = slope * 300 + intercept)

ci_300 <- predictions_300 |>
  get_ci(level = 0.95, type = "percentile")

ci_300

In [ ]:
ggplot(predictions_300, aes(x = stat)) +
  geom_histogram(bins = 20, fill = "chartreuse4", color = "white") +
  geom_vline(xintercept = c(ci_300$lower_ci, ci_300$upper_ci),
             linetype = "dashed", linewidth = 1) +
  labs(x = "Bootstrap estimates at 300 days", y = "Count") +
  theme_minimal(base_size = 16)

This is an approximate 95% confidence interval for **the height of the true line** at 300 days.

## Part 5: Near and Far From the Average

`height_ci()` gives the 95% interval for the height of the true line at any number of days, and its width.

In [ ]:
height_ci <- function(days) {
  boot_lines |>
    mutate(stat = slope * days + intercept) |>
    get_ci(level = 0.95, type = "percentile") |>
    mutate(days = days, width = upper_ci - lower_ci)
}

In [ ]:
height_ci(300)

In [ ]:
bind_rows(height_ci(230), height_ci(279), height_ci(285), height_ci(300), height_ci(310))

In [ ]:
mean(baby$`Gestational Days`)

## Part 6: A New Baby

The interval in Part 4 is for the height of the true line. A new baby's weight also has noise around the line.

In [ ]:
new_baby <- tibble(`Gestational Days` = 300)

predict(baby_model, newdata = new_baby, interval = "confidence")

In [ ]:
predict(baby_model, newdata = new_baby, interval = "prediction")

## Part 7: Check the Model

Everything above assumes the regression model holds. Look at the residual plot.

In [ ]:
residual_plot <- function(table, x, y) {
  model <- lm(table[[y]] ~ table[[x]])
  tibble(x_values = table[[x]], residuals = residuals(model)) |>
    ggplot(aes(x = x_values, y = residuals)) +
    geom_point(color = "red", alpha = 0.5) +
    geom_hline(yintercept = 0, color = "darkblue", linewidth = 2) +
    labs(x = x, title = "Residual Plot") +
    theme_minimal(base_size = 16)
}

residual_plot(baby, "Gestational Days", "Birth Weight")